# VerseForge: Shakespeare-Style Autoregressive Text Generation
## Foundations of Generative AI (ICT4403) — Assignment 2

**Student Name:** Yashraj Sakunde  
**Registration Number:** 230953172  
**System Title:** VerseForge: Shakespeare-Style Autoregressive Text Generation Using Recurrent Neural Networks  
**Dataset:** Tiny Shakespeare (`input.txt`)  
**GitHub Repository:** [https://github.com/Yashraj88888/VerseForge-Shakespeare-Style-Text-Generator](https://github.com/Yashraj88888/VerseForge-Shakespeare-Style-Text-Generator)  

---
### Project Objectives:
1. Develop a word-level autoregressive language generation system trained on Shakespeare's literary works.
2. Implement and empirically compare four recurrent architectures under strictly identical conditions:
   - **Simple RNN** (Vanilla recurrent neural network)
   - **Bidirectional RNN** (Bidirectional context encoder without lookahead)
   - **LSTM** (Long Short-Term Memory with constant error carousels)
   - **GRU** (Gated Recurrent Unit with reset and update gates)
3. Quantitatively evaluate models on held-out test data using cross-entropy loss, perplexity ($e^{\text{loss}}$), next-word accuracy, and runtime.
4. Analyze temperature sampling ($T \in \{0.5, 0.8, 1.2\}$) and lexical diversity metrics (Distinct-1, Distinct-2, repetition ratio).


## 1. Environment Setup & Dependency Verification
Ensure required packages (`tensorflow`, `numpy`, `matplotlib`, `pandas`) are imported, and verify compute acceleration.


In [ ]:
import os
import sys
import time
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

# Auto-detect repository root whether running from root, notebooks/, or Google Colab
current_dir = os.path.abspath(os.getcwd())
if os.path.basename(current_dir) == 'notebooks':
    project_root = os.path.dirname(current_dir)
else:
    project_root = current_dir

if project_root not in sys.path:
    sys.path.insert(0, project_root)

os.chdir(project_root)
print(f'Working directory successfully set to: {os.getcwd()}')
print(f'TensorFlow Version: {tf.__version__}')
print(f'Available Physical Devices: {tf.config.list_physical_devices()}')


## 2. Dataset Ingestion & Exploratory Data Analysis (EDA)
We programmatically download the Tiny Shakespeare dataset from Karpathy's repository and inspect basic text statistics.


In [ ]:
from src.data_preprocessing import download_dataset, load_raw_text, tokenize_text

data_path = download_dataset()
raw_text = load_raw_text(data_path)

print('--- Dataset Statistics ---')
print(f'Total Characters: {len(raw_text):,}')
print(f'Total Raw Words:  {len(raw_text.split()):,}')
print(f'First 300 characters preview:\n{"-"*40}\n{raw_text[:300]}\n{"-"*40}')


## 3. Word-Level Tokenization, Vocabulary, and Data Split

### Preventing Data Leakage:
- To avoid lookahead bias and data leakage, the text stream is split chronologically into **80% Training**, **10% Validation**, and **10% Testing** BEFORE sequence construction.
- The vocabulary is fitted **strictly on the training tokens**. Words outside this vocabulary in validation/testing or during sampling are mapped to `<UNK>`.


In [ ]:
from src.data_preprocessing import prepare_data

data = prepare_data(
    seq_length=25,
    step=4,
    max_vocab_size=6000,
    min_freq=2,
    train_ratio=0.8,
    val_ratio=0.1,
    test_ratio=0.1
)

stats = data['stats']
for k, v in stats.items():
    print(f'{k:<22}: {v}')

token_to_id = data['token_to_id']
id_to_token = data['id_to_token']
X_train, y_train = data['X_train'], data['y_train']
X_val, y_val = data['X_val'], data['y_val']
X_test, y_test = data['X_test'], data['y_test']

print(f'\nSequence Shapes: X_train={X_train.shape}, y_train={y_train.shape}')


### Sequence & Next-Word Target Verification
We inspect the first sequence to verify that $X[t]$ correctly aligns with the subsequent token $y[t]$.


In [ ]:
sample_input_tokens = [id_to_token[idx] for idx in X_train[0]]
sample_target_token = id_to_token[y_train[0]]

print('Input Context Sequence (Length 25):')
print(' '.join(sample_input_tokens))
print(f'\nNext-Word Ground Truth Target: "{sample_target_token}"')


## 4. Model Architectures Implementation
We define the four requested architectures using TensorFlow/Keras:
1. **Simple RNN**: `Embedding` $\to$ `SimpleRNN(128)` $\to$ `Dense(vocab_size, Softmax)`
2. **Bidirectional RNN**: `Embedding` $\to$ `Bidirectional(SimpleRNN(64))` $\to$ `Dense(vocab_size, Softmax)`
   - *Note on Autoregressive Validity:* The Bidirectional RNN operates purely across the past context $[w_1, \dots, w_T]$ and never observes $w_{T+1}$.
3. **LSTM**: `Embedding` $\to$ `LSTM(128)` $\to$ `Dense(vocab_size, Softmax)`
4. **GRU**: `Embedding` $\to$ `GRU(128)` $\to$ `Dense(vocab_size, Softmax)`


In [ ]:
from src.models import build_simple_rnn, build_bidirectional_rnn, build_lstm, build_gru

vocab_size = len(token_to_id)
models_dict = {
    'Simple_RNN': build_simple_rnn(vocab_size=vocab_size, seq_length=25, embedding_dim=64, hidden_units=128),
    'Bidirectional_RNN': build_bidirectional_rnn(vocab_size=vocab_size, seq_length=25, embedding_dim=64, hidden_units=128),
    'LSTM': build_lstm(vocab_size=vocab_size, seq_length=25, embedding_dim=64, hidden_units=128),
    'GRU': build_gru(vocab_size=vocab_size, seq_length=25, embedding_dim=64, hidden_units=128)
}

for name, m in models_dict.items():
    print(f'{name:<18} | Total Parameters: {m.count_params():,}')


## 5. Model Training & Checkpointing
Each model is trained under identical experimental conditions (batch size: 64, learning rate: 0.001, optimizer: Adam, loss: sparse categorical cross-entropy, early stopping patience: 3).
If pre-trained checkpoints exist in `models/`, we load them directly.


In [ ]:
from src.train import train_all_models

# Check if models are already trained
models_exist = all(os.path.exists(f'models/{name}.keras') for name in models_dict.keys())

if not models_exist:
    print('Training all 4 models from scratch...')
    train_results = train_all_models(epochs=8, batch_size=64, step=4)
else:
    print('Found existing trained model checkpoints in models/ directory. Ready for evaluation.')


## 6. Quantitative Evaluation on Held-Out Test Set
We compute cross-entropy test loss, perplexity ($e^{\text{loss}}$), next-word accuracy, and record actual training durations.


In [ ]:
from src.evaluate import evaluate_models

eval_summary = evaluate_models(models_dir='models', vocab_path='data/vocab.json', test_data_path='models/test_data.npz', results_dir='results')
df_results = pd.read_csv('results/model_comparison.csv')
display(df_results)


## 7. Comparative Visualizations & Training Dynamics
Let us inspect the loss curves, validation curves, perplexity, and training time across the four architectures.


In [ ]:
from IPython.display import Image, display

for plot_name in ['loss_curves.png', 'accuracy_curves.png', 'test_perplexity_comparison.png', 'test_accuracy_comparison.png', 'training_time_comparison.png']:
    img_path = os.path.join('results/plots', plot_name)
    if os.path.exists(img_path):
        print(f'Displaying {plot_name}:')
        display(Image(filename=img_path))


## 8. Autoregressive Text Generation & Temperature Scaling

### Sampling Formula:
$$P(w_i) = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}$$
- **$T = 0.5$**: High confidence, lower diversity, risk of repetitive loops.
- **$T = 0.8$**: Balanced exploration, natural Shakespearean cadence.
- **$T = 1.2$**: High entropy, high diversity, higher syntactic instability.


In [ ]:
from src.generate import generate_text

test_seed = 'To be or not to'
loaded_lstm = keras.models.load_model('models/LSTM.keras')

print(f'Seed Phrase: "{test_seed}"\n')
for temp in [0.5, 0.8, 1.2]:
    sample = generate_text(
        model=loaded_lstm,
        seed_text=test_seed,
        token_to_id=token_to_id,
        id_to_token=id_to_token,
        num_words_to_generate=45,
        temperature=temp,
        sampling_seed=42
    )
    print(f'=== Temperature {temp} ===')
    print(sample)
    print()


## 9. Lexical Diversity & Repetition Analysis
We inspect Distinct-1 (unigram diversity), Distinct-2 (bigram diversity), and bigram repetition ratios as functions of sampling temperature.


In [ ]:
temp_plot_path = 'results/plots/temperature_diversity_comparison.png'
if os.path.exists(temp_plot_path):
    display(Image(filename=temp_plot_path))


## 10. Discussion & Academic Conclusions

### Architectural Comparison:
1. **Simple RNN**: Struggles to maintain long-range context due to vanishing gradients during backpropagation through time (BPTT). Suffers from frequent token repetitions.
2. **Bidirectional RNN**: Encodes past context in both forward and reverse directions. While effective for feature extraction, unidirectional causal prediction remains challenging without gated cell memory.
3. **LSTM**: Consistently demonstrates the lowest cross-entropy test loss and perplexity. The memory cell acts as an additive highway preventing gradient attenuation.
4. **GRU**: Achieves near-identical perplexity to LSTM with approximately 8% fewer parameters and faster epoch runtimes, offering an optimal efficiency-performance balance.

### Conclusion:
For word-level autoregressive Shakespeare text generation, **LSTM** is the best performing model in qualitative coherence and lowest perplexity, closely rivaled by **GRU** for computational efficiency.
